# Aula 5: Criando um Chatbot

Este notebook apresenta um chatbot utilizando o modelo **microsoft/DialoGPT-medium**, com uma funcionalidade adicional para consulta de status de pedidos.

## Selecionando o modelo

Modelo utilizado: [microsoft/DialoGPT-medium](https://huggingface.co/microsoft/DialoGPT-medium)

O modelo é carregado uma única vez e colocado em modo de avaliação para uso em inferência.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()

print("Modelo carregado com sucesso!")

## Testando o modelo

Nesta primeira etapa, fazemos uma conversa simples com cinco interações.

**Correção importante:** usamos `max_new_tokens` em vez de `max_length`. Assim, limitamos a quantidade de tokens gerados na resposta, evitando que uma interação simples provoque uma geração excessivamente longa.

In [ ]:
# Vamos conversar por 5 interações
chat_history_ids = None
MAX_NEW_TOKENS = 50

for step in range(5):
    new_user_input_ids = tokenizer.encode(
        input(">> User: ") + tokenizer.eos_token,
        return_tensors="pt"
    )

    # Adiciona a nova entrada ao histórico da conversa
    if chat_history_ids is not None:
        bot_input_ids = torch.cat(
            [chat_history_ids, new_user_input_ids],
            dim=-1
        )
    else:
        bot_input_ids = new_user_input_ids

    # Evita ultrapassar o contexto do modelo
    bot_input_ids = bot_input_ids[:, -800:]

    with torch.no_grad():
        chat_history_ids = model.generate(
            bot_input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.eos_token_id
        )

    resposta = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print(f"DialoGPT: {resposta}")

## Criando o chatbot

Além da conversa livre, o chatbot terá uma função específica para consultar o status de pedidos.

In [ ]:
import pandas as pd

In [ ]:
dados_pedidos = {
    "numero_pedido": ["12345", "67890", "11121", "22232"],
    "status": ["Shipped", "Processing", "Delivered", "Cancelled"]
}

df_status_pedidos = pd.DataFrame(dados_pedidos)

In [ ]:
df_status_pedidos

In [ ]:
def verificar_status_pedido(numero_pedido):
    numero_pedido = str(numero_pedido).strip()

    resultado = df_status_pedidos[
        df_status_pedidos["numero_pedido"] == numero_pedido
    ]

    if not resultado.empty:
        status = resultado.iloc[0]["status"]
        return f"The status of your order {numero_pedido} is: {status}"

    return "Order number not found. Please check and try again."

In [ ]:
palavras_chave_status = [
    "order",
    "order status",
    "status of my order",
    "check my order",
    "track my order",
    "order update"
]

## Interagindo com o chatbot

Abaixo está a versão corrigida da interação.

Principais correções:
- `max_new_tokens=50` em vez de `max_length=1000`;
- `torch.no_grad()` durante a geração;
- histórico limitado para evitar crescimento indefinido;
- o modelo já foi carregado anteriormente e não é recarregado a cada mensagem;
- tratamento mais seguro da consulta de pedidos.

In [ ]:
ids_historico_chat = None

MAX_NEW_TOKENS = 50
MAX_HISTORY_TOKENS = 800

while True:
    input_usuario = input("You: ").strip()

    if input_usuario.lower() in ["exit", "quit", "stop"]:
        print("Bot: Goodbye!")
        break

    if not input_usuario:
        continue

    # Verifica se o usuário deseja consultar um pedido
    if any(keyword in input_usuario.lower() for keyword in palavras_chave_status):
        numero_pedido = input("Could you please enter your order number? ").strip()
        resposta = verificar_status_pedido(numero_pedido)

    else:
        novo_usuario_input_ids = tokenizer.encode(
            input_usuario + tokenizer.eos_token,
            return_tensors="pt"
        )

        if ids_historico_chat is not None:
            bot_input_ids = torch.cat(
                [ids_historico_chat, novo_usuario_input_ids],
                dim=-1
            )
        else:
            bot_input_ids = novo_usuario_input_ids

        # Mantém somente a parte mais recente da conversa
        bot_input_ids = bot_input_ids[:, -MAX_HISTORY_TOKENS:]

        with torch.no_grad():
            output_ids = model.generate(
                bot_input_ids,
                max_new_tokens=MAX_NEW_TOKENS,
                pad_token_id=tokenizer.eos_token_id
            )

        resposta = tokenizer.decode(
            output_ids[:, bot_input_ids.shape[-1]:][0],
            skip_special_tokens=True
        )

        # Guarda a conversa para a próxima interação
        ids_historico_chat = output_ids[:, -MAX_HISTORY_TOKENS:]

    print(f"Bot: {resposta}")

## Exemplos de uso

Conversa livre:

```text
You: hi
Bot: ...
```

Consulta de pedido:

```text
You: I want to check my order
Could you please enter your order number? 12345
Bot: The status of your order 12345 is: Shipped
```

Para encerrar:

```text
You: exit
Bot: Goodbye!
```